### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv
from langfuse.langchain import CallbackHandler
load_dotenv()
from langchain.chat_models import init_chat_model

langfuse_trace = CallbackHandler()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# model = init_chat_model("groq:qwen/qwen3-32b")
# model = init_chat_model("granite4", model_provider="ollama")
model = init_chat_model("gpt-oss", model_provider="ollama")
response = model.invoke("Why do parrots talk?")
response

d:\coding\ai_learning\krishnaik\ultimate-rag-bootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content='Parrots “talk” because they combine two things that most other animals don’t:  \n1. **The right anatomy for producing a wide range of sounds** – and  \n2. **A brain wired to learn those sounds and use them strategically in social life.**\n\n### 1. Anatomy: The Syrinx\n\n* **Syrinx = vocal organ** – located at the base of the trachea, it’s more sophisticated than our larynx. Parrots can control airflow and tension on multiple membranes independently, giving them an almost infinite spectrum of tones.\n* This hardware lets parrots generate pitches that are much lower or higher than most birds, including human speech frequencies.\n\n### 2. Brain: A “Speech‑Ready” Neural Circuit\n\n| Human | Parrot (e.g., African Grey) |\n|-------|------------------------------|\n| Broca’s area, Wernicke’s area, motor cortex | **Robust vocal‑learning network** – nuclei in the forebrain that correspond to Broca/Wernicke, plus a special song‐control system. |\n| Mirror neurons for imitation

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a given location."""
    return f"It is a sunny in {location}"

In [3]:
model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?", config={
    "callbacks": [langfuse_trace]
})
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={} response_metadata={'model': 'gpt-oss', 'created_at': '2026-07-01T12:02:44.704466106Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9691933050, 'load_duration': 1007633023, 'prompt_eval_count': 129, 'prompt_eval_duration': 2674206000, 'eval_count': 36, 'eval_duration': 5986777000, 'logprobs': None, 'model_name': 'gpt-oss', 'model_provider': 'ollama'} id='lc_run--019f1d8f-3f81-7ac1-a216-b6af83eb53dc-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '53b813e6-bf49-499d-a8ca-a12e30b019c4', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 129, 'output_tokens': 36, 'total_tokens': 165}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages, config={"callbacks": [langfuse_trace]})
print("ai_msg: ", ai_msg)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    print("tool_call: ", tool_call)
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

print("messages: ", messages)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages, config={"callbacks": [langfuse_trace]})
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

ai_msg:  content='' additional_kwargs={} response_metadata={'model': 'gpt-oss', 'created_at': '2026-07-01T12:02:54.938560978Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10208969207, 'load_duration': 1054591102, 'prompt_eval_count': 128, 'prompt_eval_duration': 2686041000, 'eval_count': 39, 'eval_duration': 6457838000, 'logprobs': None, 'model_name': 'gpt-oss', 'model_provider': 'ollama'} id='lc_run--019f1d8f-6577-7e90-bcd4-703609c94c00-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'eaefa6d9-5903-4604-8509-b31ffd3db35f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 128, 'output_tokens': 39, 'total_tokens': 167}
tool_call:  {'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'eaefa6d9-5903-4604-8509-b31ffd3db35f', 'type': 'tool_call'}
messages:  [{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss', 'created_at': 

In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss', 'created_at': '2026-07-01T12:02:54.938560978Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10208969207, 'load_duration': 1054591102, 'prompt_eval_count': 128, 'prompt_eval_duration': 2686041000, 'eval_count': 39, 'eval_duration': 6457838000, 'logprobs': None, 'model_name': 'gpt-oss', 'model_provider': 'ollama'}, id='lc_run--019f1d8f-6577-7e90-bcd4-703609c94c00-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'eaefa6d9-5903-4604-8509-b31ffd3db35f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 128, 'output_tokens': 39, 'total_tokens': 167}),
 ToolMessage(content='It is a sunny in Boston', name='get_weather', tool_call_id='eaefa6d9-5903-4604-8509-b31ffd3db35f')]